In [1]:
import os
import torch
import torch.nn.functional as F
import sys
sys.path.insert(0, "/home/ishan/Desktop/Ishans_workspace/Trans_ReID/TransReID")

from model.backbones import TransReID
from datasets import make_dataloader
from config import cfg
from utils.metrics import R1_mAP_eval

/home/ishan/miniconda3/envs/TransReID/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Create datalaoder for veri-776
cfg.merge_from_file("configs/VeRi/vit_transreid_stride.yml")
cfg.freeze()
train_loader, train_loader_normal, val_loader, num_query, num_classes, cam_num, view_num = make_dataloader(cfg)

Train directory - /home/ishan/Desktop/Ishans_workspace/datasets/VeRI-776/image_train
{0, 1, 2, 3, 4, 5, 6, 7} view_container
63 samples without viewpoint annotations
{0, 1, 2, 3, 4, 5, 6, 7} view_container
0 samples without viewpoint annotations
{0, 1, 2, 3, 4, 5, 6, 7} view_container
0 samples without viewpoint annotations
=> VeRi-776 loaded
Dataset statistics:
  ----------------------------------------
  subset   | # ids | # images | # cameras
  ----------------------------------------
  train    |   576 |    37715 |        20
  query    |   200 |     1678 |        19
  gallery  |   200 |    11579 |        19
  ----------------------------------------


In [2]:
# Create dataloader for market1501
cfg.merge_from_file("configs/Market/vit_transreid_stride.yml")
cfg.freeze()
train_loader, train_loader_normal, val_loader, num_query, num_classes, cam_num, view_num = make_dataloader(cfg)

=> Market1501 loaded
Dataset statistics:
  ----------------------------------------
  subset   | # ids | # images | # cameras
  ----------------------------------------
  train    |   751 |    12936 |         6
  query    |   750 |     3368 |         6
  gallery  |   751 |    15913 |         6
  ----------------------------------------


In [3]:
model = TransReID(
    img_size=(256, 256), 
    stride_size=16, 
    drop_rate=0.0, 
    attn_drop_rate=0.0, 
    drop_path_rate=0.1, 
    camera=0, 
    view=0, 
    local_feature=False, 
    sie_xishu=1.5,
    qkv_bias=True
)

# load weights
weights = "/home/ishan/Desktop/Ishans_workspace/pretrained_weights/vit_base_veri.pth"
weight_dict = torch.load(weights, map_location='cpu')
model.load_state_dict(weight_dict)

using stride: 16, and patch number is num_y16 * num_x16
using drop_out rate is : 0.0
using attn_drop_out rate is : 0.0
using drop_path rate is : 0.1


/tmp/ipykernel_70940/2183730193.py:16: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  weight_dict = torch.load(weights, map_location='cpu')


<All keys matched successfully>

In [ ]:
# Inference loop 
from tqdm.auto import tqdm

device = torch.device("cuda")
model = model.to(device)
model.eval()

img_path_list = []
evaluator = R1_mAP_eval(num_query=num_query, max_rank=50, feat_norm=cfg.TEST.FEAT_NORM)
evaluator.reset()

with torch.no_grad():
    for n_iter, (img, pid, camid, camids, target_view, imgpath) in tqdm(
        enumerate(val_loader),
        total=len(val_loader),
        desc="Evaluating on test set..."
    ):

        img = img.to(device)
        camids = camids.to(device)
        target_view = target_view.to(device)
        feat = model(img, cam_label=camids, view_label=target_view)
        evaluator.update((feat, pid, camid))
        img_path_list.extend(imgpath)

Evaluating on test set...:   5%|▌         | 2/38 [00:24<07:20, 12.22s/it]

In [5]:
cmc, mAP, _, _, _, _, _ = evaluator.compute()

The test feature is normalized
=> Computing DistMat with euclidean_distance


/home/ishan/Desktop/Ishans_workspace/Trans_ReID/TransReID/utils/metrics.py:12: UserWarning: This overload of addmm_ is deprecated:
	addmm_(Number beta, Number alpha, Tensor mat1, Tensor mat2)
Consider using one of the following signatures instead:
	addmm_(Tensor mat1, Tensor mat2, *, Number beta = 1, Number alpha = 1) (Triggered internally at ../torch/csrc/utils/python_arg_parser.cpp:1642.)
  dist_mat.addmm_(1, -2, qf, gf.t())


In [8]:
print(f"Rank-1: {cmc[0]:.4f}")
print(f"Rank-5: {cmc[4]:.4f}")
print(f"Rank-10: {cmc[9]:.4f}")
print(f"mAP: {mAP:.4f}")

Rank-1: 0.9648
Rank-5: 0.9863
Rank-10: 0.9911
mAP: 0.7893
